In [1]:
# ============================================================
# STEP 6 — Inference
# Loads the saved model and classifies individual emails.
# ============================================================

import os
import torch
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from google.colab import drive

# ============================================================
# 6.0  Mount Drive + Folders
# ============================================================
drive.mount("/content/drive", force_remount=True)

SAVE_DIR = "/content/drive/MyDrive/PhishingClassifier"
MAX_LEN  = 256
device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for folder in [SAVE_DIR, f"{SAVE_DIR}/model"]:
    os.makedirs(folder, exist_ok=True)

# ============================================================
# 6.1  Load Model
# ============================================================
print("Loading model...")
model     = AutoModelForSequenceClassification.from_pretrained(f"{SAVE_DIR}/model").to(device)
tokenizer = AutoTokenizer.from_pretrained(f"{SAVE_DIR}/model")
model.eval()
print(f"  Ready on {device}\n")

# ============================================================
# 6.2  Predict Function
# ============================================================
def predict(text: str, threshold: float = 0.5) -> dict:
    text = str(text).strip()
    if not text:
        text = "empty"
    inputs = tokenizer(
        text,
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )
    ids  = inputs["input_ids"].to(device)
    mask = inputs["attention_mask"].to(device)
    with torch.no_grad():
        out   = model(input_ids=ids, attention_mask=mask)
        probs = F.softmax(out.logits, dim=-1)[0]
    safe_score     = round(probs[0].item(), 4)
    phishing_score = round(probs[1].item(), 4)
    verdict        = "PHISHING" if phishing_score >= threshold else "SAFE"
    return {
        "verdict":         verdict,
        "phishing_score":  phishing_score,
        "safe_score":      safe_score,
        "confidence":      round(max(safe_score, phishing_score), 4)
    }

# ============================================================
# 6.3  Test Emails
# ============================================================
tests = [
    {"expected": "PHISHING",
     "text": "URGENT: Your PayPal account has been limited! "
             "Click http://paypa1-secure.ru/verify to restore access now."},
    {"expected": "PHISHING",
     "text": "Dear Customer, we detected unusual activity on your bank account. "
             "Confirm your details at http://secure-hsbc-verify.com immediately."},
    {"expected": "SAFE",
     "text": "Hi team, reminder about Thursday 3pm sync in Conference Room B."},
    {"expected": "SAFE",
     "text": "Your Amazon order #483-291 has shipped. Arrives Friday. Track at amazon.com."},
    {"expected": "SAFE",
     "text": "Monthly newsletter from Midnight Reads — top 10 thrillers this week."}
]

print("=" * 60)
print("INFERENCE RESULTS")
print("=" * 60)

correct = 0
for t in tests:
    r     = predict(t["text"])
    match = "✓" if r["verdict"] == t["expected"] else "✗"
    correct += 1 if r["verdict"] == t["expected"] else 0
    print(f"\n  {match} Expected  : {t['expected']}")
    print(f"    Verdict    : {r['verdict']}  (confidence: {r['confidence']:.2%})")
    print(f"    Phishing   : {r['phishing_score']:.4f}  |  Safe: {r['safe_score']:.4f}")
    print(f"    Text       : {t['text'][:65]}...")

print(f"\nScore : {correct}/{len(tests)}")
print("Step 6 complete. Proceed to 07_api_server.py")

Mounted at /content/drive
Loading model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  Ready on cuda

INFERENCE RESULTS

  ✓ Expected  : PHISHING
    Verdict    : PHISHING  (confidence: 100.00%)
    Phishing   : 1.0000  |  Safe: 0.0000
    Text       : URGENT: Your PayPal account has been limited! Click http://paypa1...

  ✓ Expected  : PHISHING
    Verdict    : PHISHING  (confidence: 99.99%)
    Phishing   : 0.9999  |  Safe: 0.0001
    Text       : Dear Customer, we detected unusual activity on your bank account....

  ✓ Expected  : SAFE
    Verdict    : SAFE  (confidence: 100.00%)
    Phishing   : 0.0000  |  Safe: 1.0000
    Text       : Hi team, reminder about Thursday 3pm sync in Conference Room B....

  ✓ Expected  : SAFE
    Verdict    : SAFE  (confidence: 99.98%)
    Phishing   : 0.0002  |  Safe: 0.9998
    Text       : Your Amazon order #483-291 has shipped. Arrives Friday. Track at ...

  ✓ Expected  : SAFE
    Verdict    : SAFE  (confidence: 99.27%)
    Phishing   : 0.0073  |  Safe: 0.9927
    Text       : Monthly newsletter from Midnight Reads — top 10 thril